In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [2]:
load_dotenv()  # Load environment variables from .env file

True

In [3]:
model=ChatOpenAI()

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages] 
    # The `messages` field is a list of `BaseMessage` objects, and the `add_messages` annotation indicates that this field will be used to add messages to the state graph.

In [6]:
def chat_node(state: ChatState) -> ChatState:
    messages = state['messages']
    response=model.invoke(messages)

    return {'messages':[response]}


In [15]:
checkpointer=MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

workflow=graph.compile(checkpointer=checkpointer)

# workflow.invoke({'messages':[HumanMessage(content="Hello, how are you?")]})['messages'][-1].content

In [19]:
thread_id='1'

while True:
    user_message=input("You: ")
    print("User:", user_message)  # Debugging line to print the user message
    if user_message.strip().lower() in ["exit", "quit","bye"]:
        print("Exiting the chat. Goodbye!")
        break

    config={'configurable':{'thread_id':thread_id}}
    response=workflow.invoke({'messages':[HumanMessage(content=user_message)]},config=config)['messages'][-1].content

    print("Chatbot:", response)

User: What's my name?
Chatbot: Your name is MAV.
User: exit
Exiting the chat. Goodbye!
